In [ ]:
import sys
sys.path.append("../../")
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# data = pd.read_csv(r"../data/stroke_pre/stroke.csv")
# data = pd.read_csv('data/diabetes_pre/diabetes.csv')
data = pd.read_csv('../data/wids_pre/wids.csv')

In [ ]:
data.info()

In [ ]:
from callmefair.search.fair_search import BiasSearch

dfbias = BiasSearch(data, 'DiagPeriodL90D', ['patient_age', 'race', 'payer', 'state_privileged', 'neighborhood_education'], n_threads=32) 
#dfbias = BiasSearch(data, 'stroke', ['age', 'gender', 'ever_married', 'Residence_type'], n_threads=10) 
#dfbias = BiasSearch(data, 'readmitted', ['age', 'gender', 'race'], n_threads=16)

In [ ]:
tab_lr, printable = dfbias.evaluate_average(model_name='lr')
printable

In [ ]:
tab_xgb, printable = dfbias.evaluate_average(model_name='xgb')
printable

In [ ]:
tab_mlp, _ = dfbias.evaluate_average(model_name='mlp')
tab_cat, _ = dfbias.evaluate_average(model_name='cat')

table, printable = dfbias.evaluate_combination_average('age','gender')
printable

In [ ]:
table, printable = dfbias.evaluate_combination_average('age','race')
printable

In [ ]:
table, printable = dfbias.evaluate_combination_average('ever_married','gender')
printable

In [ ]:
table, printable = dfbias.evaluate_combinations()
printable

In [ ]:
tab1, tab2 = tab_lr, table[1:]
tab_final = [*tab1, *tab2]
print(tab_final)

In [ ]:
import plotly.graph_objects as go


def process_array(data, group_color):
    """Process 2D array into plot components with border coloring"""
    attributes = [d[0] for d in data[1:]]
    raw_scores = [d[1] for d in data[1:]]
    normalized = [d[2] for d in data[1:]]
    
    # Determine border colors based on normalized scores
    border_colors = ['red' if n == 1 else 'green' for n in normalized]
    
    return {
        'attributes': attributes,
        'raw': raw_scores,
        'border_colors': border_colors,
        'color': group_color
    }

# Process both datasets
group1 = process_array(tab_lr, '#1f77b4')  # Blue
group2 = process_array(table, '#ff7f0e')  # Orange

# Calculate x positions with gap between groups
x1 = list(range(len(group1['attributes'])))
x2 = [len(group1['attributes']) + 1 + i for i in range(len(group2['attributes']))]

# Create bar traces with conditional borders
trace1 = go.Bar(
    x=x1,
    y=group1['raw'],
    name='Group 1',
    marker=dict(
        color=group1['color'],
        line=dict(
            color=group1['border_colors'],
            width=3  # Thicker border for visibility
        )
    )
)

trace2 = go.Bar(
    x=x2,
    y=group2['raw'],
    name='Group 2',
    marker=dict(
        color=group2['color'],
        line=dict(
            color=group2['border_colors'],
            width=3
        )
    )
)

# Create figure
fig = go.Figure([trace1, trace2])

# Calculate axis settings
tickvals = x1 + x2
ticktext = group1['attributes'] + group2['attributes']
separator_pos = len(group1['attributes'])  # Position between groups

# Update layout with visual separation
fig.update_layout(
    xaxis=dict(
        tickvals=tickvals,
        ticktext=ticktext,
        title='Attributes',
        showgrid=False
    ),
    yaxis=dict(title='Raw Fairness Score'),
    title='Fairness Analysis with Normalization Indicators',
    bargap=0.25,
    shapes=[dict(
        type='line',
        xref='x',
        yref='paper',
        x0=separator_pos,
        y0=0,
        x1=separator_pos,
        y1=1,
        line=dict(color='gray', width=2, dash='dot')
    )]
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

def create_fairness_visualization(attributes, raw_scores, normalized_scores, title="Attribute Fairness Impact"):
    """
    Create a visualization of fairness scores with actual data values.
    
    Parameters:
    -----------
    attributes : list
        List of attribute names
    raw_scores : list of lists
        Raw fairness scores for each attribute across multiple measurement points
    normalized_scores : list of lists
        Normalized fairness scores corresponding to raw scores
    title : str
        Title for the plot
    
    Returns:
    --------
    fig : plotly.graph_objects.Figure
        The plotly figure object
    """
    # Create figure
    fig = go.Figure()
    
    # Calculate average raw scores for sorting
    avg_raw_scores = [sum(scores)/len(scores) for scores in raw_scores]
    
    # Create sorting indices based on average raw scores
    sort_indices = np.argsort(avg_raw_scores)
    
    # Sort attributes and scores
    sorted_attributes = [attributes[i] for i in sort_indices]
    sorted_raw_scores = [raw_scores[i] for i in sort_indices]
    sorted_norm_scores = [normalized_scores[i] for i in sort_indices]
    
    # Set y-axis labels (attributes)
    y_values = sorted_attributes
    
    # Add scatter points for each attribute
    for i, (attr, raw_list, norm_list) in enumerate(zip(sorted_attributes, sorted_raw_scores, sorted_norm_scores)):
        # Use actual data points instead of generated ones
        x_values = raw_list
        color_values = norm_list
        point_count = len(x_values)
        
        # Add the points with a scatter plot
        fig.add_trace(go.Scatter(
            x=x_values,
            y=[i] * point_count,
            mode='markers',
            marker=dict(
                size=10,
                color=color_values,
                colorscale='RdBu',
                line=dict(width=0),
                opacity=0.7
            ),
            showlegend=False,
            hoverinfo='text',
            text=[f"{attr}: {raw:.3f}" for raw in x_values]
        ))
        
        # Add a violin plot for density visualization using actual data
        fig.add_trace(go.Violin(
            x=x_values,
            y=[i] * point_count,
            box_visible=False,
            points=False,
            line_color='rgba(0, 0, 0, 0)',
            fillcolor='rgba(180, 180, 180, 0.3)',
            width=0.6,
            side='both',
            orientation='h',
            showlegend=False,
            hoverinfo='none'
        ))
        
        # Add a larger point to highlight the mean raw score
        mean_raw = sum(raw_list) / len(raw_list)
        mean_norm = sum(norm_list) / len(norm_list)
        fig.add_trace(go.Scatter(
            x=[mean_raw],
            y=[i],
            mode='markers',
            marker=dict(
                size=14,
                color=mean_norm,
                colorscale='RdBu',
                line=dict(width=1, color='black'),
                opacity=1.0
            ),
            showlegend=False,
            hoverinfo='text',
            text=f"{attr} (mean): {mean_raw:.3f}"
        ))
    
    # Configure layout
    fig.update_layout(
        title=title,
        xaxis=dict(
            title='Fairness Score (impact on model output)',
            zeroline=False,
            range=[min([min(scores) for scores in sorted_raw_scores]) - 0.2, 
                  max([max(scores) for scores in sorted_raw_scores]) + 0.2]
        ),
        yaxis=dict(
            title='',
            tickvals=list(range(len(y_values))),
            ticktext=y_values,
            showgrid=False
        ),
        height=max(400, len(sorted_attributes) * 40),  # Adjust height based on number of attributes
        width=900,
        coloraxis=dict(
            colorbar=dict(
                title="Feature value",
                tickvals=[0, 1],
                ticktext=["Low", "High"]
            ),
            colorscale='RdBu'
        ),
        plot_bgcolor='rgba(255, 255, 255, 1)',
        margin=dict(l=150)  # Ensure enough space for attribute names
    )
    
    return fig

attributes = [att[0] for att in tab_lr[1:]]
raw_scores = [[lr[1], mlp[1], xgb[1], cat[1]] for lr, mlp, xgb, cat in zip(tab_lr[1:], tab_mlp[1:], tab_xgb[1:], tab_cat[1:])]
normalized_scores = [[lr[2], mlp[2], xgb[2], cat[2]] for lr, mlp, xgb, cat in zip(tab_lr[1:], tab_mlp[1:], tab_xgb[1:], tab_cat[1:])]

fig = create_fairness_visualization(attributes, raw_scores, normalized_scores, 
                                  title="Classifier Fairness Impact by Attribute")
fig.show()